# N2 ground state with 300 x 300 determinants

I searched for the lowest N2 ground-state energy I could reach with at most 300 alpha and 300 beta CI strings (Qiskit Fall Fest 2026 Warsaw, Track 4). The submitted solution is one rotated orbital basis plus one 300 x 300 product subspace, with energy -108.938468 Ha.

Grader rules that shaped the result: a submission holds at most 300 strings per spin; the wavefunction lives in the product subspace spanned by the alpha strings times the beta strings; the grader recomputes the energy from my files with `solve_sci` and requires my claimed energy to match; no spin constraint is imposed, so the alpha and beta sets may differ; the recomputed energy is a lowest eigenvalue, hence variational (it can only sit above the exact ground state, never below).

My assumptions: 2 frozen core orbitals, 26 active orbitals with 5 alpha + 5 beta electrons; classical selected-CI search only, no QPU time used.

In [1]:
import json
from pathlib import Path

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ROOT = Path(".").resolve()
if not (ROOT / "submission").exists():
    ROOT = ROOT.parent  # kernel runs in notebooks/
SUB = ROOT / "submission"
FIG = ROOT / "figures"
FIG.mkdir(exist_ok=True)
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 150})

U = np.load(SUB / "rotation.npy")
h = np.load(SUB / "hcore.npy")
g = np.load(SUB / "eri.npy")
ch = np.load(SUB / "canonical_hcore.npy")
cg = np.load(SUB / "canonical_eri.npy")
sa = np.load(SUB / "ci_strs_a.npy")
sb = np.load(SUB / "ci_strs_b.npy")
manifest = json.loads((SUB / "manifest.json").read_text())

n = U.shape[0]
print("U shape:", U.shape, " det:", float(np.linalg.det(U)))
print("max |U^T U - I|:", float(np.max(np.abs(U.T @ U - np.eye(n)))))
hr = U.T @ ch @ U
print("max |h - U^T h_can U|:", float(np.max(np.abs(hr - h))))
t = np.einsum("ap,abcd->pbcd", U, cg)
t = np.einsum("bq,pbcd->pqcd", U, t)
t = np.einsum("cr,pqcd->pqrd", U, t)
gr = np.einsum("ds,pqrd->pqrs", U, t)
print("max |g - rotated g_can|:", float(np.max(np.abs(gr - g))))
bits = lambda s: ((np.asarray(s)[:, None] >> np.arange(26)) & 1).sum(axis=1)
print("strings:", len(sa), "x", len(sb),
      " popcount ok:", bool((bits(sa) == 5).all() and (bits(sb) == 5).all()))
print("submitted energy:", manifest["energy"])
print("full grader recompute (2-6 min): python scripts/verify_rotation.py submission")

U shape: (26, 26)  det: 1.0000000000000029
max |U^T U - I|: 3.552713678800501e-15
max |h - U^T h_can U|: 0.0
max |g - rotated g_can|: 0.0
strings: 300 x 300  popcount ok: True
submitted energy: -108.938468
full grader recompute (2-6 min): python scripts/verify_rotation.py submission


## Energy ladder

Each step is a saved candidate energy on the path I took, from HF to the submitted state. CCSD is a non-variational reference and FCI a literature value; both are shown for scale.

In [2]:
HF = -108.330583
LAB = -108.789          # quoted lab baseline
HFSD = -108.862724       # HF + singles + doubles, capped at 300 per spin
SEL = -108.914953        # selected CI in canonical orbitals
NAT = -108.919881        # first natural-orbital step
ORB = -108.924135        # orbital optimisation
IND = -108.938278        # independent alpha/beta sets (fixes U)
SUBM = -108.938468       # basin hopping: submitted
CCSD = -108.909521
FCI = -108.98203

labels = ["HF", "lab baseline", "HF+SD", "selected CI",
          "natural orbitals", "orbital optimisation",
          "independent alpha/beta", "submitted"]
vals = [HF, LAB, HFSD, SEL, NAT, ORB, IND, SUBM]
gains = [(HF - v) * 1000.0 for v in vals]

fig, ax = plt.subplots(figsize=(8.5, 4.6))
y = np.arange(len(labels))
ax.barh(y, gains, color="#1f77b4")
ax.set_yticks(y, labels)
ax.invert_yaxis()
ax.set_xlabel("Gain over HF (mHa)")
ax.set_title("Energy ladder")
for yi, gv, v in zip(y, gains, vals):
    ax.text(gv, yi, f"  {v:.6f}", va="center", fontsize=8)
ax.axvline((HF - CCSD) * 1000.0, color="gray", ls="--", label="CCSD -108.909521")
ax.axvline((HF - FCI) * 1000.0, color="gray", ls=":", label="FCI -108.98203")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "energy_ladder.png", bbox_inches="tight")
plt.close(fig)
print("saved figures/energy_ladder.png")

saved figures/energy_ladder.png


## Optimisation history

Each point is an accepted candidate (a new best at the time) from my two searches. Both minimise the same energy; the second run allows the alpha and beta string sets to differ. CCSD is shown for scale.

In [3]:
from n2sqd.report import accepted_proposals

searches = {
    "orbital optimisation": ROOT / "runs" / "B_1d9fa1ba.jsonl",
    "independent alpha/beta": ROOT / "runs" / "B_23ed77e0.jsonl",
}
colors = ["#1f77b4", "#6baed6"]

fig, (ax_t, ax_i) = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for (label, path), color in zip(searches.items(), colors):
    acc = accepted_proposals(path)
    hours = [t / 3600.0 for t, _, _ in acc]
    iters = [k for _, k, _ in acc]
    energies = [e for _, _, e in acc]
    ax_t.plot(hours, energies, "o-", ms=4, color=color, label=label)
    ax_i.plot(iters, energies, "o-", ms=4, color=color, label=label)
ax_t.set_xlabel("Hours since run start")
ax_i.set_xlabel("Proposal iteration")
ax_t.set_ylabel("Energy (Ha)")
for ax in (ax_t, ax_i):
    ax.axhline(CCSD, color="gray", ls="--", lw=1, label="CCSD -108.909521")
ax_t.set_title("Energy vs wall time")
ax_i.set_title("Energy vs step")
ax_i.legend(fontsize=8, loc="best")
fig.suptitle("Optimisation history")
fig.tight_layout()
fig.savefig(FIG / "optimisation_history.png", bbox_inches="tight")
plt.close(fig)
print("saved figures/optimisation_history.png")

saved figures/optimisation_history.png


## Rotation and occupations

The submitted basis differs from the canonical one by a single orthogonal rotation. The occupations below weight each stored string by its amplitude in the submitted state.

In [4]:
fig, ax = plt.subplots(figsize=(4.6, 4.0))
im = ax.imshow(np.abs(U), vmin=0, vmax=1)
ax.set_xlabel("Optimised orbital")
ax.set_ylabel("Canonical orbital")
ax.set_title("Rotation |U|")
fig.colorbar(im, ax=ax, label="|U|")
fig.tight_layout()
fig.savefig(FIG / "rotation.png", bbox_inches="tight")
plt.close(fig)
print("saved figures/rotation.png")

saved figures/rotation.png


In [5]:
from n2sqd.report import marginals, orbital_occupations

z = np.load(ROOT / "results" /
              "cand_Em108p938468_20261024T221615Z_B_hopB_k30s202_n30_energy_ind.npz",
              allow_pickle=False)
C = np.asarray(z["amplitudes"], dtype=float)
za = np.asarray(z["strings_a"], dtype=np.int64)
zb = np.asarray(z["strings_b"], dtype=np.int64)
wa, wb = marginals(C)
occ_a = orbital_occupations(za, wa)
occ_b = orbital_occupations(zb, wb)
print("weight sums:", float(wa.sum()), float(wb.sum()),
      " electron sums:", float(occ_a.sum()), float(occ_b.sum()))

fig, ax = plt.subplots(figsize=(9, 3.4))
x = np.arange(26)
ax.bar(x - 0.2, occ_a, width=0.4, label="alpha", color="#1f77b4")
ax.bar(x + 0.2, occ_b, width=0.4, label="beta", color="#6baed6")
ax.set_xlabel("Orbital")
ax.set_ylabel("Mean occupation")
ax.set_title("Orbital occupations")
ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "occupations.png", bbox_inches="tight")
plt.close(fig)
print("saved figures/occupations.png")

weight sums: 1.0 1.0  electron sums: 4.999999999999998 5.000000000000001


saved figures/occupations.png


## What the numbers say

I reran the same selection with the orbitals held fixed and reached -108.915506, so the rotated basis is worth about 23 mHa. The submitted -108.938468 sits 28.9 mHa below CCSD (-108.909521, a non-variational reference) and 43.6 mHa above the FCI literature value (-108.98203).

In [6]:
print("Figures written:")
for p in sorted(FIG.glob("*.png")):
    print(f"- figures/{p.name}")

Figures written:
- figures/energy_ladder.png
- figures/occupations.png
- figures/optimisation_history.png
- figures/rotation.png
